# Modules 17–18: Prompt Engineering & Retrieval-Augmented Generation (RAG) Assessment
**Assessment Code**: M18-A1  
**Student Name**: Dhruv Munjpara  
**Enrollment / Student ID**: 8849162891  
**Course**: Data Science & AI Master Program — TOPS Technologies  

---

## Executive Summary & Notebook Index
This master notebook contains the complete solution for the official **Prompt Engineering & Retrieval-Augmented Generation (RAG) Assessment (M18-A1)** centered on a real-world **QuickBite Food Delivery Platform** AI assistant scenario.

- [Section A — Concept Application (Scenarios S1 to S6)](#section-a)
- [Section B — Practical Coding Tasks (Tasks 1 to 4)](#section-b)
  - [Task 1: Structured Prompt Builder + Input Validation](#task-1)
  - [Task 2: Few-Shot Complaint Classifier Prompt Builder](#task-2)
  - [Task 3: Semantic Search Over Restaurant FAQs with FAISS](#task-3)
  - [Task 4: Complete RAG Policy Q&A Pipeline](#task-4)
- [Section C — Mini Capstone Project: Interactive Support Console](#section-c)
- [Section D — AI-Augmented Learning (Prompts, Bug Fixes & Code Diffs)](#section-d)


<a id='section-a'></a>
# Section A — Concept Application (Scenarios S1 – S6)

### Scenario S1: Prompt Clarity & Consistency in Support Chatbots
- **Unclear vs Clear Prompt**: Unclear prompts leave decision criteria vague, causing inconsistent refund decisions. Clear prompts specify support agent persona, deterministic IF-THEN refund rules, and strict length/format limits.
- **Change 1 (Decision Rule Matrix)**: Added explicit rules for missing items, late deliveries, and escalation criteria to eliminate subjective LLM interpretation.
- **Change 2 (Output Schema & Max Length)**: Added max 80 words constraint and 2-sentence response structure to prevent unauthorized promises.

### Scenario S2: Zero-Shot vs Few-Shot Complaint Classifier
- **Comparison**: Zero-shot uses minimal context but struggles with ambiguous complaints. Few-shot uses 4–8 concrete exemplar pairs, significantly improving edge-case precision within context limits.
- **Justification**: Few-shot is chosen because 4–8 balanced examples consume only ~200 tokens while providing explicit patterns for classifying ambiguous complaints across 4 categories (`Late Delivery`, `Wrong Item`, `Missing Item`, `Poor Quality`).

### Scenario S3: Chain-of-Thought (CoT) Route Optimization
- **CoT Improvement**: Decomposes multi-variable stop sequence calculation into intermediate reasoning steps (priority grouping -> traffic delay calculation -> distance optimization).
- **Limitation**: Higher token generation increases API latency and inference cost for real-time dispatch systems.

### Scenario S4: RAG vs Fine-Tuning for 150-Page Policy Handbook
- **Justification for RAG**: (1) Quarterly handbook updates require seconds in RAG (vector database update) vs expensive GPU retraining in Fine-Tuning; (2) RAG guarantees zero hallucinations by citing exact policy paragraphs.
- **Fine-Tuning Superiority**: Fine-tuning is better for teaching specialized domain vocabulary or custom XML/JSON syntax formatting.

### Scenario S5: Chunking Parameters for 500 PDF Menus
- **Chunk Size & Overlap**: Increased chunk size to 250 words with 50-word overlap to ensure dish names and price blocks stay unified across chunk boundaries.
- **Trade-off**: Slightly higher vector storage and context token usage per query.

### Scenario S6: Vector Store Comparison (FAISS vs ChromaDB)
- **FAISS vs ChromaDB**: FAISS offers ultra-fast in-memory vector search but lacks easy metadata deletion. ChromaDB supports native document updates and metadata filtering (`status == 'active'`).
- **Required Step**: Automated metadata pre-filtering and deletion synchronization pipeline.

<a id='section-b'></a>
# Section B — Practical Coding Tasks

<a id='task-1'></a>
### Task 1: Structured Prompt Builder + Input Validation

In [ ]:
# Task 1 Code
ALLOWED_ISSUE_TYPES = {'late delivery', 'missing item', 'wrong item'}

def validate_issue_type(issue_type: str) -> str:
    normalized = issue_type.strip().lower()
    if normalized not in ALLOWED_ISSUE_TYPES:
        raise ValueError(f"Invalid issue_type '{issue_type}'. Must be one of {sorted(ALLOWED_ISSUE_TYPES)}")
    return normalized

def generate_support_prompt(customer_name: str, order_id: str, issue_type: str):
    try:
        valid_issue = validate_issue_type(issue_type)
        system_prompt = (
            "System: You are an AI Support Agent for QuickBite Food Delivery. "
            "Assist customers professionally. Max response length: 80 words."
        )
        user_prompt = (
            f"Customer: {customer_name} | Order: {order_id} | Issue: {valid_issue.title()}"
        )
        print(f"--- SYSTEM PROMPT ---\n{system_prompt}")
        print(f"--- USER PROMPT ---\n{user_prompt}\n")
    except ValueError as e:
        print(f"❌ ERROR: {e}\n")

# Test cases
generate_support_prompt("Rahul Sharma", "QB-98421", "late delivery")
generate_support_prompt("Priya Patel", "QB-77104", "missing item")
generate_support_prompt("Amit Kumar", "QB-55412", "rude driver")

<a id='task-2'></a>
### Task 2: Few-Shot Complaint Classifier Prompt Builder

In [ ]:
# Task 2 Code
FEW_SHOT_EXAMPLES = [
    {"input": "My pizza arrived 50 minutes late.", "output": "Late Delivery"},
    {"input": "I received a Chicken Burger instead of Paneer Wrap.", "output": "Wrong Item"},
    {"input": "The delivery bag was missing the garlic bread.", "output": "Missing Item"},
    {"input": "The soup spilled all over and was cold.", "output": "Poor Quality"}
]

def add_example(text: str, label: str):
    FEW_SHOT_EXAMPLES.append({"input": text, "output": label})

def build_few_shot_prompt(complaint: str) -> str:
    prompt = "System: Classify into 'Late Delivery', 'Wrong Item', 'Missing Item', or 'Poor Quality'.\n\n"
    for ex in FEW_SHOT_EXAMPLES:
        prompt += f"Input: \"{ex['input']}\"\nOutput: {ex['output']}\n\n"
    prompt += f"Input: \"{complaint}\"\nOutput:"
    return prompt

print(build_few_shot_prompt("My order was missing 2 tacos."))

<a id='task-3'></a>
### Task 3: Semantic Search Over Restaurant FAQs with FAISS

In [ ]:
# Task 3 Code
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

FAQ_DATASET = [
    "FAQ 1: Standard delivery time is 30 to 45 minutes.",
    "FAQ 2: Orders can be cancelled within 60 seconds of submission.",
    "FAQ 3: Full refunds are issued for missing items within 3 to 5 days.",
    "FAQ 4: Out of stock items will trigger a call for substitution.",
    "FAQ 5: Contact 24/7 customer support via app Help live chat.",
    "FAQ 6: Free delivery minimum order is ₹200 for regular members."
]

st_model = SentenceTransformer('all-MiniLM-L6-v2')
faq_embs = st_model.encode(FAQ_DATASET).astype(np.float32)

faiss_idx = faiss.IndexFlatL2(faq_embs.shape[1])
faiss_idx.add(faq_embs)

def search_faq(query: str, k: int = 2):
    q_v = st_model.encode([query]).astype(np.float32)
    distances, indices = faiss_idx.search(q_v, k)
    return [(FAQ_DATASET[idx], float(distances[0][r])) for r, idx in enumerate(indices[0])]

print("Search Result for 'How do I get money back?':")
for faq, dist in search_faq("How do I get money back?", k=2):
    print(f" Dist: {dist:.4f} | {faq}")

<a id='task-4'></a>
### Task 4: Complete RAG Policy Q&A Pipeline

In [ ]:
# Task 4 Code
POLICY_DOC = """
QuickBite Policy Handbook: Customers can cancel free within 60 seconds before restaurant confirmation. 
Refunds for missing items are credited to original payment within 3 to 5 business days or instant wallet cash. 
Wrong item deliveries receive 100% refund plus ₹50 voucher upon photo proof within 2 hours. 
Delays over 45 minutes receive ₹100 delay credit compensation.
"""

words = POLICY_DOC.split()
policy_chunks = [" ".join(words[i:i+40]) for i in range(0, len(words), 30)]
p_embs = st_model.encode(policy_chunks).astype(np.float32)
p_faiss = faiss.IndexFlatL2(p_embs.shape[1])
p_faiss.add(p_embs)

def build_rag_prompt(query: str, k: int = 2):
    q_v = st_model.encode([query]).astype(np.float32)
    D, I = p_faiss.search(q_v, k)
    retrieved = [policy_chunks[idx] for idx in I[0]]
    
    prompt = (
        "System: Answer ONLY using provided context below. If missing, say 'I don't know.'\n\n"
        f"Context: {' '.join(retrieved)}\n\n"
        f"Question: {query}\n\nAnswer:"
    )
    return prompt

print(build_rag_prompt("What is the refund policy for missing items?"))

<a id='section-c'></a>
# Section C — Mini Capstone Project: Interactive Helpdesk Support Console
*(See standalone file `Section_C_Mini_Capstone/mini_capstone_helpdesk.py` for full CLI app)*

In [ ]:
# Execute Non-Interactive Test of Mini Capstone
import subprocess
res = subprocess.run(["python", "Section_C_Mini_Capstone/mini_capstone_helpdesk.py", "--test"], capture_output=True, text=True)
print(res.stdout)

<a id='section-d'></a>
# Section D — AI-Augmented Learning (Report & Code Fixes)

### AI Prompt Submitted:
> *"Write a Python script using SentenceTransformers, FAISS, and PyPDF2 that loads a refund policy PDF, chunks text, builds a vector index, retrieves top chunks, and creates an LLM RAG prompt with a while loop."*

### 4 Critical Bug Fixes Implemented:
1. **Word-Boundary Chunking**: Fixed character slicing `[i:i+200]` cutting words mid-sentence.
2. **Index Boundary Guard**: Added `k = min(k, len(chunks))` to stop FAISS out-of-bounds crashes.
3. **Fallback Instruction**: Injected strict context grounding and fallback constraint (`"Say 'I don't know' if missing"`).
4. **Sanitized Input Loop**: Replaced rigid `user_input == "quit"` with `user_input.strip().lower() in ["quit", "exit", ""]`.